In [ ]:
# INITIAL IMPORTS



In [ ]:
# BARCODE AND INSERT EXTRACTION

import edlib
import mappy as mp
import sys
from collections import namedtuple

# Define Hit object
Hit = namedtuple('Hit', ['r_st', 'r_en', 'strand', 'trans_strand', 'score'])

def get_best_hit_edlib(query_seq, target_seq, threshold_pct=0.30):
    """
    Replaces parasail with edlib for infix alignment.
    
    Args:
        query_seq: Primer/Barcode sequence
        target_seq: The long read
        threshold_pct: Max allowed error rate (0.30 = 30% differences allowed)
    """
    best_hit = None
    best_ed = float('inf') # Lower edit distance is better
    
    # We define strands as: 
    # 1 = Query matches target as-is
    # -1 = Reverse Complement of Query matches target
    # Note: 'trans_strand' is just for compatibility with your existing logic logic
    
    candidates = [
        (query_seq, 1), 
        (mp.revcomp(query_seq), -1)
    ]
    
    query_len = len(query_seq)
    
    # Calculate max allowed errors (mismatches + indels)
    # e.g., 20bp query * 0.30 = max 6 errors
    max_dist = int(query_len * threshold_pct)

    for seq, strand_val in candidates:
        # mode="HW" (Infix): Find best location of 'seq' inside 'target_seq'
        # task="locations": Return start/end indices
        # k=max_dist: Stop searching if errors exceed this (HUGE speedup)
        result = edlib.align(seq, target_seq, mode="HW", task="locations", k=max_dist)
        
        if result['editDistance'] == -1:
            continue # No hit within threshold

        # If this is the best hit so far (lowest edit distance)
        if result['editDistance'] < best_ed:
            best_ed = result['editDistance']
            
            # Edlib can return multiple optimal locations, we take the first one
            # result['locations'] is a list of tuples [(start, end), ...]
            loc = result['locations'][0]
            
            best_hit = Hit(
                r_st=loc[0],
                r_en=loc[1] + 1, # Edlib is inclusive, Python slice is exclusive
                strand=strand_val,
                trans_strand="+", # Legacy placeholder
                score=result['editDistance'] # Note: Lower is better now
            )
            
    return best_hit

def extract_region(read_seq, flank_5p, flank_3p):
    """
    Extracts sequence between two flanking sequences using Edlib.
    """
    # 1. Find the flanks
    h5 = get_best_hit_edlib(flank_5p, read_seq)
    h3 = get_best_hit_edlib(flank_3p, read_seq)

    if not h5 or not h3:
        return None, "Flanks not found"

    # 2. Check Orientation Consistency
    if h5.strand != h3.strand:
        return None, "Flank orientation mismatch"

    # 3. Extract based on strand
    # Forward Strand (Both matched as Forward)
    if h5.strand == 1:
        # h5 should appear BEFORE h3
        if h5.r_en >= h3.r_st:
            return None, "Negative distance (Overlap/Swap)"
        
        # Extract the middle
        return read_seq[h5.r_en : h3.r_st], "+"

    # Reverse Strand (Both matched as Reverse Complement)
    else:
        # If the read is RC, the RC(5'BC) appears at the END of the read
        # and RC(3'BC) appears at the START of the read.
        # So physically in the read, h3 comes before h5.
        if h3.r_en >= h5.r_st:
            return None, "Negative distance (RC Overlap/Swap)"
            
        rc_segment = read_seq[h3.r_en : h5.r_st]
        return mp.revcomp(rc_segment), "-"

def process_fastq(fastq_path, bc_5p, bc_3p, ins_5p, ins_3p, min_length_fastq=1000):
    print(f"Processing: {fastq_path}")
    print("Read_ID\tBC_Status\tBC_Seq\tIns_Status\tIns_Seq")

    pairs = []
    i = 0
    
    # Use mappy to read fastq (fast!)
    for name, seq, qual in mp.fastx_read(fastq_path):
        if len(seq) < min_length_fastq:
            continue

        i += 1
        
        # Extract Barcode
        bc_seq, bc_status = extract_region(seq, bc_5p, bc_3p)
        
        # Extract Insert
        ins_seq, ins_status = extract_region(seq, ins_5p, ins_3p)

        bc_out = bc_seq if bc_seq else "NA"
        ins_out = ins_seq if ins_seq else "NA"

        pairs.append([bc_out, ins_out])
        
        # Optional: Print progress
        if i % 1000 == 0:
            print(f"Processed {i} reads...", file=sys.stderr)

    return pairs